# YOLO Training - Building Defect Detection (Merged Dataset)

## Dataset: 6,875 images
- Train: 4,692 images
- Validation: 1,785 images
- Test: 398 images

## Classes: 15
- general_damage, cracks, mold, water_damage, structural_damage
- electrical_issues, plumbing_issues, roofing_damage
- window_damage, door_damage, floor_damage, wall_damage
- ceiling_damage, hvac_issues, insulation_issues

## Cell 1: Setup Environment

In [ ]:
# Install Ultralytics YOLO
!pip install ultralytics -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Cell 2: Upload Dataset ZIP

**Before running this cell:**
1. ZIP the merged dataset:
   - On your computer: `yolo_dataset_merged_final/` folder
   - Create ZIP: `yolo_dataset_merged_final.zip`
2. Upload to Google Drive: `/MyDrive/yolo_dataset_merged_final.zip`

**Or upload directly in Colab (slower):**

In [ ]:
import os
import zipfile
from pathlib import Path

# UPDATE THIS PATH to match your Google Drive folder
ZIP_PATH = '/content/drive/MyDrive/YOLO_Training/yolo_dataset_merged_final.zip'

# Extract dataset
if os.path.exists(ZIP_PATH):
    print(f"Found ZIP: {ZIP_PATH}")
    print(f"Size: {os.path.getsize(ZIP_PATH) / 1024**2:.1f} MB")
    
    print("\nExtracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
    
    print("\nDataset extracted!")
    
    # Find where data.yaml actually is
    print("\nSearching for data.yaml...")
    data_yaml_found = False
    for yaml_file in Path('/content/dataset').rglob('data.yaml'):
        print(f"   Found: {yaml_file}")
        data_yaml_found = True
        
    if not data_yaml_found:
        print("   ERROR: data.yaml not found!")
    
    # Count images
    print("\nSearching for image folders...")
    for img_dir in Path('/content/dataset').rglob('images'):
        if img_dir.is_dir():
            count = len(list(img_dir.glob('*')))
            if count > 0:
                print(f"   {img_dir}: {count:,} files")
    
    # Try to count from expected locations
    try:
        train_path = Path('/content/dataset/yolo_dataset_merged_final/train/images')
        val_path = Path('/content/dataset/yolo_dataset_merged_final/val/images')
        test_path = Path('/content/dataset/yolo_dataset_merged_final/test/images')
        
        if train_path.exists():
            train_images = len(list(train_path.glob('*')))
            val_images = len(list(val_path.glob('*')))
            test_images = len(list(test_path.glob('*')))
            
            print(f"\nDataset Stats:")
            print(f"   Train: {train_images:,} images")
            print(f"   Val: {val_images:,} images")
            print(f"   Test: {test_images:,} images")
            print(f"   TOTAL: {train_images + val_images + test_images:,} images")
        else:
            print("\nWARNING: Expected folder structure not found!")
            print("The ZIP may have extracted with an extra folder level.")
            print("Check the paths above and update Cell 3 accordingly.")
    except Exception as e:
        print(f"\nNote: {e}")
        
else:
    print(f"ERROR: ZIP not found at {ZIP_PATH}")
    print("\nPlease:")
    print("1. Verify the ZIP is uploaded to Google Drive")
    print("2. Update ZIP_PATH in this cell to match your folder structure")
    print("3. Re-run this cell")

## Cell 3: Train YOLO Model

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path

# Auto-find data.yaml location
print("="*70)
print("FINDING DATASET")
print("="*70)

# Search for data.yaml
data_yaml_path = None
for yaml_file in Path('/content/dataset').rglob('data.yaml'):
    data_yaml_path = str(yaml_file)
    print(f"\nFound data.yaml at: {data_yaml_path}")
    break

if not data_yaml_path:
    print("\nERROR: data.yaml not found!")
    print("Please check Cell 2 output and verify extraction completed successfully.")
    raise FileNotFoundError("data.yaml not found in /content/dataset")

# Training configuration
DATA_YAML = data_yaml_path
MODEL = 'yolo11n.pt'
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16

print("\n" + "="*70)
print("YOLO TRAINING STARTED")
print("="*70)

print(f"\nConfiguration:")
print(f"   Data: {DATA_YAML}")
print(f"   Model: {MODEL}")
print(f"   Epochs: {EPOCHS}")
print(f"   Image Size: {IMG_SIZE}")
print(f"   Batch Size: {BATCH_SIZE}")

# Load model (will auto-download ~6MB if not exists)
print(f"\nDownloading {MODEL}...")
model = YOLO(MODEL)
print("Model loaded successfully!")

# Train
print(f"\nStarting training... (this will take 2-4 hours on T4 GPU)\n")

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,  # GPU
    project='runs/train',
    name='merged_model_v1',
    patience=20,
    save=True,
    plots=True,
    verbose=True,
    val=True
)

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)

print("\nBest model saved to: runs/train/merged_model_v1/weights/best.pt")
print("Results saved to: runs/train/merged_model_v1/")

## Cell 4: Validate Model

In [ ]:
# Validate best model
best_model = YOLO('runs/train/merged_model_v1/weights/best.pt')

print("="*70)
print("MODEL VALIDATION")
print("="*70)

# Validate on test set
metrics = best_model.val(data=DATA_YAML, split='test')

print(f"\nValidation Results:")
print(f"   mAP50: {metrics.box.map50:.3f}")
print(f"   mAP50-95: {metrics.box.map:.3f}")
print(f"   Precision: {metrics.box.mp:.3f}")
print(f"   Recall: {metrics.box.mr:.3f}")

# Per-class metrics
print(f"\nPer-Class AP50:")
class_names = ['general_damage', 'cracks', 'mold', 'water_damage', 'structural_damage',
               'electrical_issues', 'plumbing_issues', 'roofing_damage', 'window_damage',
               'door_damage', 'floor_damage', 'wall_damage', 'ceiling_damage', 'hvac_issues',
               'insulation_issues']

for i, name in enumerate(class_names):
    if i < len(metrics.box.ap50):
        print(f"   {name}: {metrics.box.ap50[i]:.3f}")

## Cell 5: Download Trained Model

In [ ]:
import shutil
from google.colab import files

# Create ZIP of results
print("Creating results ZIP...")
shutil.make_archive('/content/yolo_trained_model', 'zip', 'runs/train/merged_model_v1')

print("\nZIP created: yolo_trained_model.zip")
print(f"Size: {os.path.getsize('/content/yolo_trained_model.zip') / 1024**2:.1f} MB")

# Save to Google Drive (recommended - faster)
shutil.copy('/content/yolo_trained_model.zip', '/content/drive/MyDrive/yolo_trained_model.zip')
print("\nSaved to Google Drive: /MyDrive/yolo_trained_model.zip")

# Also download directly (backup)
print("\nStarting download to your computer...")
files.download('/content/yolo_trained_model.zip')

print("\nDONE! Trained model ready for deployment.")

## Next Steps

After downloading the trained model:

1. **Extract the ZIP** to get `best.pt`
2. **Compare with existing model:**
   - Current: `best_model_final.pt`
   - New: `best.pt` from training
3. **Convert to ONNX** if new model is better:
   ```python
   from ultralytics import YOLO
   model = YOLO('best.pt')
   model.export(format='onnx')
   ```
4. **Deploy to production:**
   - Update `.env.local`: `YOLO_MODEL_PATH=../../best_v2.onnx`
   - Restart dev server
   - Test building defect detection